# Instrument Tracker - Training Pipeline

This notebook runs the full pipeline: **Catalog -> Dataset -> Train -> Evaluate**

All logic lives in the scripts (`data_prep/`, `train/`, `eval/`).  
The notebook just calls functions and shows results.

**Prerequisites:** SurgicalInstruments app running at localhost:3000, conda env `sam3`

In [ ]:
import sys, json, random
from pathlib import Path

COMPONENT = Path.cwd()
DATA = COMPONENT / 'data'
API_URL = 'http://localhost:3000'

sys.path.insert(0, str(COMPONENT / 'data_prep'))
sys.path.insert(0, str(COMPONENT))

import urllib.request
health = json.loads(urllib.request.urlopen(f'{API_URL}/api/health', timeout=5).read())
print(f'Catalog: {health.get("instruments","?")} instruments, {health.get("images","?")} images')

---
## Step 1 - Sync from Catalog

Pulls taxonomy, kits, images, and videos (.webm) from the SurgicalInstruments app.  
**Incremental:** only downloads NEW or UPDATED instruments (tracked via `sync_manifest.json`).

In [ ]:
from sync_from_catalog import sync_taxonomy, plan_sync, download_instruments

instruments, kits, families = sync_taxonomy(API_URL, str(DATA))
print(f'Instruments: {len(instruments)}  |  Kits: {len(kits)}  |  Families: {len(families)}')

In [ ]:
to_download, to_skip, manifest = plan_sync(instruments, str(DATA / 'sync_manifest.json'))
print(f'To download: {len(to_download)}  |  Skipped: {len(to_skip)} (unchanged)')

if to_download:
    stats = download_instruments(API_URL, str(DATA), to_download, manifest)
    print(f'Images: {stats["images"]}  Videos: {stats["videos"]}  Frames: {stats["frames"]}')
else:
    print('Everything up to date.')

---
## Step 2 - Verify Downloads

Checks every file is valid. Corrupt files are auto-deleted. Shows sample thumbnails.

In [ ]:
from sync_from_catalog import verify_downloads
import cv2, matplotlib.pyplot as plt

v = verify_downloads(str(DATA))
print(f'Valid images: {v["valid_images"]}  |  Valid videos: {v["valid_videos"]}  |  Corrupt: {len(v["corrupt"])}')

all_imgs = list(DATA.glob('exemplars/**/*.jpg')) + list(DATA.glob('exemplars/**/*.png'))
if all_imgs:
    samples = random.sample(all_imgs, min(4, len(all_imgs)))
    fig, axes = plt.subplots(1, len(samples), figsize=(4*len(samples), 4))
    if len(samples) == 1: axes = [axes]
    for ax, p in zip(axes, samples):
        ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
        ax.set_title(p.parent.name[:25], fontsize=8); ax.axis('off')
    plt.tight_layout(); plt.show()

---
## Step 3 - Deduplicate Names

Same instrument, different names across kits -> merged into one class.  
Different COLOR -> same class. Different SIZE/SHAPE -> different class.  
Review `data/dedup_log.json` after this step.

In [ ]:
from run_pipeline import deduplicate_names

class_map, dedup_log = deduplicate_names(instruments, str(DATA))
print(f'Original: {dedup_log["total_instruments"]}  |  Unique classes: {dedup_log["unique_classes"]}  |  Merged: {dedup_log["merged_groups"]}')

if dedup_log['merges']:
    print('\nMerged groups:')
    for key, names in list(dedup_log['merges'].items())[:5]:
        print(f'  {key}  <-  {names}')

---
## Step 4 - Build YOLO Dataset

Collects images, auto-labels (contour detection), splits train/val/test, exports YOLO format.

In [ ]:
from run_pipeline import build_yolo_dataset

ys = build_yolo_dataset(class_map, str(DATA))
print(f'Total: {ys["total"]}  |  Train: {ys["train"]}  |  Val: {ys["val"]}  |  Test: {ys["test"]}  |  Classes: {ys["n_classes"]}')

---
## Step 5 - Visual Sanity Check

In [ ]:
from run_pipeline import show_samples
show_samples(class_map, str(DATA), n_classes=6, n_per_class=3)

---
## Step 6 - Train YOLOv8-seg

Adjust parameters below. ~15-20h on RTX 4090.

In [ ]:
EPOCHS, IMGSZ, BATCH, MODEL = 50, 1024, 4, 'yolov8m-seg.pt'
DATA_YAML = str(DATA / 'yolo_dataset' / 'data.yaml')

import torch
print(f'CUDA: {torch.cuda.is_available()}', end='')
if torch.cuda.is_available():
    print(f'  |  GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f}GB')
else:
    print()
print(f'Classes: {ys["n_classes"]}  |  Train: {ys["train"]} images  |  Ready to train.')

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
            device=0, project=str(COMPONENT/'runs'), name='detector', exist_ok=True)
print(f'Done. Weights: {COMPONENT/"runs"/"detector"/"weights"/"best.pt"}')

---
## Step 7 - Evaluate

In [ ]:
BEST = COMPONENT / 'runs' / 'detector' / 'weights' / 'best.pt'
if not BEST.exists():
    print('No weights yet. Run Step 6 first.')
else:
    m = YOLO(str(BEST)).val(data=DATA_YAML, split='test', imgsz=IMGSZ, batch=BATCH, device=0, verbose=False)
    print(f'mAP@50: {m.box.map50:.4f}  |  mAP@50-95: {m.box.map:.4f}  |  Prec: {m.box.mp:.4f}  |  Recall: {m.box.mr:.4f}')

---
## Step 8 - Test on Sample Image

In [ ]:
if BEST.exists():
    test_imgs = list((DATA/'yolo_dataset'/'images'/'test').glob('*')) or list((DATA/'yolo_dataset'/'images'/'val').glob('*'))
    if test_imgs:
        s = random.choice(test_imgs)
        r = YOLO(str(BEST)).predict(str(s), imgsz=IMGSZ, conf=0.25, verbose=False)
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
        a1.imshow(cv2.cvtColor(cv2.imread(str(s)), cv2.COLOR_BGR2RGB)); a1.set_title('Original'); a1.axis('off')
        a2.imshow(cv2.cvtColor(r[0].plot(), cv2.COLOR_BGR2RGB)); a2.set_title('Detections'); a2.axis('off')
        plt.tight_layout(); plt.show()